# DSCI 6883 - Descriptive Analytics & Data Integrity
## Discussion 2.3 & 2.4 Follow-Up: Types, Names, and Joins

**Fall 2026 | Module 2 | Fernando Rubio Garcia**

In [0]:
# Housekeeping - run this once, then forget about it.
#
# Several cells below are *designed* to raise an error, so you can see what that error
# actually looks like. By default Jupyter stops a "Run All" at the first error it hits,
# which would stop this notebook partway through. The line below turns that off.
#
# It is wrapped in try/except because it only applies to Jupyter. If you are running
# somewhere else (Databricks, a plain .py script), it does nothing and that is fine.
try:
    get_ipython().kernel._abort_queues = lambda *a, **kw: None
except Exception:
    pass

### Where this came from

In the 2.3 & 2.4 discussion I asked you to point at the parts of merging and cleaning that did not land. The posts clustered around three things, and **every section below answers one of them**. The outline says whose.

The best observation in the thread came from Kara, and she almost buried it in a reply: *"I noticed on the video the type was listed as 'object.' In the example, when I imported from excel I never got the generic 'object.'"* Kara, you were right, and it was not you. The video and your laptop are running different versions of pandas, and pandas changed how it labels text between them. Section 1 explains the change, and it turns out to make Matei's question easier to answer, not harder.

If your part is in here and it still does not click, reply in the discussion with the **section number and the cell** where it stopped making sense.

### How to use this notebook

1. **Run every cell yourself**, top to bottom, with `Shift + Enter`.
2. **Predict the output before you run a cell.**
3. **Do the `Your Turn` exercises** before opening the answers.
4. Cells marked `SUPPOSED TO FAIL` are meant to raise an error. Read the error, then keep going.
5. The **assignment is at the end**. It uses the real Superstore file from 2.3 and 2.4.

Everything in this notebook is generated in memory. It does not read or write any files, so it runs anywhere, including Databricks.

### Outline

| Section | Topic | From the post by |
|---|---|---|
| 1 | "Why don't I ever see `object`?" | Kara |
| 2 | String or category? A rule you can write down | Matei, Kara |
| 3 | Column names: dots, brackets, and snake_case | Kara, Scott, Jacob |
| 4 | It ran. Did it work? Checking a merge | Scott, Matei |
| 5 | `axis=0`, `axis=1`, and why `merge` feels bolted on | Scott |
| 6 | Deciding what to change: the decision log | Jacob |
| | **Assignment** | |

Sections 1 and 2 are one question asked two ways: *what type should this column be, and how do I justify it?* Section 4 is the one I most want you to take into your project, because a bad merge does not error. It just quietly gives you the wrong number of rows.

---
## Setup: build the practice data

Run the next two cells once. They build small stand-ins for the two datasets from 2.3 and 2.4: a Superstore-style orders table with its People sheet, and an Olympics-style athletes table with its `noc_regions` lookup. They are shaped like the real files, including the real files' problems.

In [0]:
%pip install openpyxl

In [0]:
import warnings

import numpy as np
import pandas as pd

print("pandas version:", pd.__version__)

rng = np.random.default_rng(6883)
n = 20_000

regions = ["Central US", "Eastern US", "Western US", "Southern US", "Canada",
           "Western Europe", "Southern Europe", "Oceania", "Southeastern Asia", "Caribbean"]
subcats = {
    "Furniture":       ["Bookcases", "Chairs", "Furnishings", "Tables"],
    "Office Supplies": ["Appliances", "Art", "Binders", "Envelopes", "Fasteners",
                        "Labels", "Paper", "Storage", "Supplies"],
    "Technology":      ["Accessories", "Copiers", "Machines", "Phones"],
}
sub_to_cat = {s: c for c, subs in subcats.items() for s in subs}
all_subs = list(sub_to_cat)

sub = rng.choice(all_subs, n)
orders = pd.DataFrame({
    "Row ID":         np.arange(1, n + 1),
    "Order ID":       [f"CA-{2014 + (i // 2) % 2}-{100000 + i // 2}" for i in range(n)],  # 2 lines per order
    "Order Date":     pd.Timestamp("2014-01-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D"),
    "Ship Mode":      rng.choice(["Standard Class", "Second Class", "First Class", "Same Day"], n,
                                 p=[.6, .2, .15, .05]),
    "Customer Name":  [f"Customer {i:04d}" for i in rng.integers(0, 800, n)],
    "Segment":        rng.choice(["Consumer", "Corporate", "Home Office"], n, p=[.52, .30, .18]),
    "Region":         rng.choice(regions, n),
    "Category":       [sub_to_cat[s] for s in sub],
    "Sub-Category":   sub,
    "Product Name":   [f"{s} model {k}" for s, k in zip(sub, rng.integers(0, 150, n))],
    "Sales":          np.round(rng.gamma(2.0, 120.0, n), 2),
    "Quantity":       rng.integers(1, 10, n),
    "Order Priority": rng.choice(["Medium", "High", "Critical", "Low"], n, p=[.57, .30, .08, .05]),
})

# The People sheet: one manager per region. Like the real file, it splits Canada in two,
# while the orders only ever say "Canada".
people_regions = [r for r in regions if r != "Canada"] + ["Eastern Canada", "Western Canada"]
people = pd.DataFrame({
    "Person": [f"Manager {i:02d}" for i in range(len(people_regions))],
    "Region": people_regions,
})

print("orders:", orders.shape, "  people:", people.shape)

In [0]:
# Olympics-style tables. Athlete rows carry a 3-letter NOC code; noc_regions maps code -> country.
athletes = pd.DataFrame({
    "Name": ["A. Tan", "B. Lim", "C. Ng", "D. Smith", "E. Jones", "F. Brown",
             "G. Rossi", "H. Bianchi", "I. Sato", "J. Ito"],
    "NOC":  ["SGP", "SGP", "SGP", "USA", "USA", "GBR", "ITA", "ITA", "JPN", "JPN"],
    "Year": [2008, 2012, 2016, 2016, 2012, 2016, 2008, 2016, 2012, 2016],
})

# Like the real noc_regions.csv, Singapore is listed as SIN, not SGP.
noc_regions = pd.DataFrame({
    "NOC":    ["SIN", "USA", "GBR", "ITA", "JPN"],
    "region": ["Singapore", "USA", "UK", "Italy", "Japan"],
})

print("athletes:", athletes.shape, "  noc_regions:", noc_regions.shape)

---
# 1. "Why don't I ever see `object`?"

**From Kara's reply to Matei:** *"I thought a 'text' column would be the same thing as string, but I noticed on the video the type was listed as 'object.' In the example, when I imported from excel I never got the generic 'object.' Regardless, I guess my question is when is it ok to leave a column type as 'object'?"*

Two questions there. The first one has a short answer. Look at the version number the setup cell printed, then run this:

In [0]:
orders.dtypes

What you see depends on your version:

| pandas version | A column of text shows up as |
|---|---|
| 1.x and 2.x (the Udemy videos, and many work laptops) | `object` |
| **3.0 and later** (a fresh install this year) | `str` |

pandas 3 came out at the start of 2026 and changed the default: text columns now get their own type, `str`, instead of the catch-all `object`. So the video shows `object` and Kara's Excel import shows `str`, and **both mean "this column holds text."** Nothing about the Superstore file changed.

If you are on the older version you will keep seeing `object`, and everything in this notebook still works for you. I tested it on both.

In [0]:
%pip show pandas

### 1.1 What `object` actually means

`object` is not a text type. It means *"pandas could not find one type that fits, so each cell holds whatever Python object it was given."* In older pandas, text landed there because there was nowhere better for it. Numbers mixed with words landed there too, and you could not tell the two apart from the dtype.

That is why the change matters. **In pandas 3, if you see `object`, something is mixed.** It stopped being the normal label for text and became a warning.

Here is the most common way it happens: an Excel column where some cells are numbers and some are text. Postal codes are the classic case.

In [0]:
import io

# Build a tiny Excel file in memory: US zips typed as numbers, plus a placeholder and a Canadian code.
buffer = io.BytesIO()
pd.DataFrame({
    "city":        ["Wichita", "Overland Park", "Unknown", "Ottawa", "Salina"],
    "postal_code": [67052, 66210, "TBD", "K1A 0B1", 67401],
}).to_excel(buffer, index=False)
buffer.seek(0)

addresses = pd.read_excel(buffer)
print(addresses.dtypes)

`postal_code` is `object` in every version, because it really is mixed. The dtype tells you *that* something is mixed, not *what*. This one line tells you what:

In [0]:
addresses["postal_code"].map(type).value_counts()

Three integers and two strings. `.map(type)` asks each cell what it is, and `value_counts()` tallies the answers. **Run it on every `object` column you meet.** On older pandas it is the only way to tell "all text" from "mixed", and on pandas 3 it tells you what the mix is.

### 1.2 When is it OK to leave a column as `object`?

Kara's real question. My answer:

- **All text, on old pandas:** fine. That is just how pandas 1 and 2 store text. If you want the pandas 3 behavior on an older version, `.astype("string")` gives you a strict text type that rejects anything that is not a string or missing.
- **Mixed, anywhere:** not fine. A mixed column behaves differently depending on which row you touch. Sorting it can raise an error, `.str` methods return `NaN` for the numbers, and comparing it to a value silently misses the rows stored as the other type. Decide what it should be and convert all of it.
- **Should be a number or a date:** not fine, whatever the version. That was Section 4 of the 2.2 follow-up, and it is still the most common problem in real files.

The next cell shows the "silently misses" problem, because it is the one that gets people.

In [0]:
# How many addresses are in zip 67052?
print("compare to the number 67052: ", (addresses["postal_code"] == 67052).sum())
print("compare to the string '67052':", (addresses["postal_code"] == "67052").sum())

# The fix: decide the column is text (postal codes are labels, not quantities), convert all of it.
addresses["postal_code"] = addresses["postal_code"].astype("string")
print("\nafter converting to string:")
print(addresses["postal_code"].map(type).value_counts())
print("compare to the string '67052':", (addresses["postal_code"] == "67052").sum())

Before the fix, the answer depended on whether you typed `67052` or `"67052"`, and one of them said zero. Neither errored. After the fix the column has one type and one answer.

> **Why postal codes are text:** you never add or average them, and as numbers they lose leading zeros (`07052` becomes `7052`). In the real Superstore file, `Postal Code` loads as a float and 449 codes have already lost their leading zero. The rule from 2.2 still holds: **if you would never do arithmetic on it, it is not a number**, however much it looks like one.

---
# 2. String or category? A rule you can write down

**From Matei's post:** *"I was able to convert the date columns to datetime and columns with repeated values to category, but I am still not completely sure when a text column should use the string type instead of the category type. More examples comparing these data types and explaining how the choice affects the DataFrame would help."*

Matei, "columns with repeated values to category" is the rule the Udemy course teaches, and it is half right. Almost every text column has repeated values. `Order ID` repeats (one order has several lines). `Customer Name` repeats. Neither should be a category. So you need a better test, and to get one you need to know what a category actually is.

### 2.1 What a category is

A `category` column stores each distinct value **once**, in a small lookup list, and then stores a small integer code for every row. `"Office Supplies"` is kept one time, and 12,000 rows just hold the number `1`.

In [0]:
orders["Segment"]

In [0]:
seg = orders["Segment"].astype("category")

print("the lookup list:", list(seg.cat.categories))
print("the codes for the first 8 rows:", seg.cat.codes.head(8).tolist())
print("the values for the first 8 rows:", seg.head(8).tolist())

That design has four consequences, and each one is a reason to choose it or avoid it.

### 2.2 Consequence 1: memory, which depends on how many distinct values there are

In [0]:
def memory_mb(series):
    return series.memory_usage(deep=True) / 1e6

rows = []
for col in ["Segment", "Region", "Sub-Category", "Product Name", "Customer Name", "Order ID"]:
    s = orders[col]
    rows.append({
        "column":        col,
        "distinct":      s.nunique(),
        "distinct/rows": round(s.nunique() / len(s), 3),
        "as text (MB)":  round(memory_mb(s), 3),
        "as category (MB)": round(memory_mb(s.astype("category")), 3),
    })

pd.DataFrame(rows)

Read the `distinct/rows` column against the two memory columns. Your exact megabytes depend on your pandas version; the pattern does not.

- `Segment`, `Region`, `Sub-Category`: a handful of distinct values across 20,000 rows. Category is a small fraction of the size.
- `Order ID`: half the rows are distinct. The lookup list is nearly as big as the column, and category saves little or nothing.
- `Customer Name` is the interesting one. 800 names in 20,000 rows, so category would save a lot of memory. It is still a string, and 2.4 explains why.

So the first half of a better rule: **category pays off when the number of distinct values is small compared with the number of rows.** A few dozen distinct values in thousands of rows, yes. Thousands of distinct values, no.

Memory is the least important of the four consequences, though. On a 20,000-row file nobody cares about a few megabytes. The next three change how your DataFrame *behaves*.

### 2.3 Consequence 2: an ordered category sorts in business order

`Order Priority` has four values with a real order. As text, pandas sorts them alphabetically, which is meaningless:

In [0]:
print("as text, sorted:")
print(orders["Order Priority"].value_counts().sort_index())

In [0]:
priority_type = pd.CategoricalDtype(["Low", "Medium", "High", "Critical"], ordered=True)
orders["Order Priority"] = orders["Order Priority"].astype(priority_type)

print("as an ordered category, sorted:")
print(orders["Order Priority"].value_counts().sort_index())

print("\norders more urgent than Medium:", (orders["Order Priority"] > "Medium").sum())

Low, Medium, High, Critical, in that order, and `> "Medium"` now means "more urgent than Medium" instead of "later in the alphabet." Every table and every chart built from this column will come out in the right order without you sorting it by hand. **This is the strongest reason to use a category**, and it applies to anything with a natural order: sizes, ratings, education levels, age bands, risk levels.

### 2.4 Consequence 3: a category only accepts the values on its list

A category is a closed set. Try to put in a value that is not on the list and pandas refuses:

In [0]:
# SUPPOSED TO FAIL -- TypeError: Cannot setitem on a Categorical with a new category
orders.loc[0, "Order Priority"] = "Hihg"

That is a feature when the set really is closed. `Order Priority` should never contain `"Hihg"`, and now it cannot.

It is a nuisance when the set is open. New customers and new products show up all the time, which is why `Customer Name` stays a string even though category would shrink it. If `Customer Name` were a category, every new customer would need the category list updated first. That is the second half of the rule: **category is for a fixed set of labels; string is for values where new ones legitimately appear.**

### 2.5 Consequence 4: a category remembers values that are no longer there

This is the one that surprises people. Filter a category column and the empty categories are still listed:

In [0]:
seg = orders["Segment"].astype("category")
consumers_only = seg[seg == "Consumer"]

print("value_counts after filtering to Consumer:")
print(consumers_only.value_counts())

`Corporate` and `Home Office` show up with a count of zero, even though we filtered them out. The category list belongs to the column, not to the rows. A bar chart built from this will draw two empty bars, and a `groupby` can show empty groups.

Sometimes you want that. A monthly report that shows "0 critical orders this month" is more honest than one where the row disappears. When you do not want it, drop them:

In [0]:
print(consumers_only.cat.remove_unused_categories().value_counts())

> **On `groupby`:** whether empty categories show up in a `groupby` is controlled by `observed=`, and the default flipped between pandas versions. Write it explicitly, `df.groupby("col", observed=True)`, and your code behaves the same everywhere.

### 2.6 The rule

Here it is as four questions, in order. Stop at the first "yes."

| # | Question | If yes |
|---|---|---|
| 1 | Is it really a number or a date? | Convert it: `pd.to_numeric`, `pd.to_datetime`. It is not text at all. |
| 2 | Is it an identifier or free text? (IDs, names, descriptions, addresses, postal codes) | **String.** New values appear, and the values are not groups you compare. |
| 3 | Is it a small, fixed set of labels you will group, filter, or count by? | **Category.** If the labels have a natural order, an **ordered** category. |
| 4 | Not sure? | **String.** It is the safe default. A category is a constraint plus an optimization, so add it when you know you want both. |

Matei, the reason you could not write a justification is that "it has repeated values" is not a reason. These four questions are. Here they are applied to every column in the orders table:

| Column | Type | Reason (the sentence for your markdown cell) |
|---|---|---|
| `Row ID` | int | A row counter. Keep as a number, but never analyze it. |
| `Order ID` | string | An identifier. New orders appear constantly, and half the values are distinct. |
| `Order Date` | datetime | A date. Loads as a datetime from Excel already; from a CSV you would convert it. |
| `Ship Mode` | category | Four fixed options, grouped and counted. Arguably ordered by speed. |
| `Customer Name` | string | An identifier. An open set, and 800 distinct values. |
| `Segment` | category | Three fixed labels, and we compare them constantly. |
| `Region` | category | Ten fixed labels, the main grouping column. |
| `Category` | category | Three fixed labels. |
| `Sub-Category` | category | Seventeen fixed labels, and each belongs to one `Category`. |
| `Product Name` | string | Free text. Thousands of distinct values, and new products appear. |
| `Sales` | float | A quantity you add and average. |
| `Quantity` | int | A count. |
| `Order Priority` | **ordered** category | Four fixed labels with a real order: Low < Medium < High < Critical. |

Notice that the table has no row that says "because it has repeated values." Every reason is about **what the column is and what you will do with it.**

In [0]:
# Applying the table.
closed_sets = ["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]
open_sets   = ["Order ID", "Customer Name", "Product Name"]

before_mb = orders.memory_usage(deep=True).sum() / 1e6

typed = orders.copy()
typed[closed_sets] = typed[closed_sets].astype("category")
typed[open_sets]   = typed[open_sets].astype("string")
# Order Priority is already the ordered category from 2.3.

after_mb = typed.memory_usage(deep=True).sum() / 1e6
print(typed.dtypes)
print(f"\nmemory: {before_mb:.1f} MB -> {after_mb:.1f} MB")

#### Your Turn 2.1 - Type the Olympics columns

The real `athlete_events.csv` from 2.3 has these text columns. For each, pick **string, category, ordered category, or something else**, and write the one-sentence reason. Do it in a markdown cell before opening the answer.

`Name`, `Sex`, `Team`, `NOC`, `Games` (e.g. `"2016 Summer"`), `Season`, `City`, `Sport`, `Event`, `Medal`

<details>
<summary><b>Show answer</b></summary>

| Column | Type | Reason |
|---|---|---|
| `Name` | string | An identifier, 130,000+ distinct values. |
| `Sex` | category | Two fixed values. |
| `Team` | string | Over 1,000 distinct values, many of them boat names and `"-1"`, `"-2"` suffixes. It needs cleaning before it is a set of anything. |
| `NOC` | category | About 230 fixed codes, the join key and a grouping column. |
| `Games` | **something else** | It is two columns glued together. Split it into `Year` (already its own column) and `Season`, and you no longer need it. |
| `Season` | category | Two values, Summer and Winter. |
| `City` | category | A few dozen host cities, a fixed historical set. String is also defensible; you will rarely group by it. |
| `Sport` | category | About 65 fixed values, a main grouping column. |
| `Event` | category or string | About 765 distinct values in 271,000 rows, so category saves memory, but you will rarely group by it. Either is fine **if the reason is written down**. |
| `Medal` | **ordered** category | Bronze < Silver < Gold. And note that `NaN` here means *no medal*, not *missing*, which you should say in your markdown. |

The two worth a second look are `Games` and `Medal`. `Games` is a reminder that question 1 is really "what *is* this column?", and sometimes the answer is "a column that should not exist." `Medal` is the best example in the course of a missing value that is not missing.

</details>

---
# 3. Column names: dots, brackets, and snake_case

**From Kara's post:** *"if column has a good name, you can just do like `df.State.value_counts()` but if the column has a poor name, you must put it in quotes... Should we change the column names to have '_' in place of ' ' and '-'?"*

**From Scott's reply:** *"I went looking for a pandas convention and surprisingly there isn't one. The best thing I could find is a recommendation to follow Python and use snake_case."*

**From Jacob's post:** *"Python is always super tricky when it comes to naming conventions and especially in these dataframes."*

Short answer: **yes, rename, and Scott found the right convention.** pandas has no official style, so everyone falls back on Python's: lowercase words joined by underscores, `order_date`, `sub_category`. Here is why it is worth doing, and a one-liner that does it.

### 3.1 Why the dot works for some names and not others

`df.Region` is not really "looking up a column." It is Python attribute access, the same dot as in `"text".upper()`. pandas lets columns answer to it as a convenience, but only when the name is a legal Python name: letters, digits, and underscores, not starting with a digit.

`Sub-Category` is not a legal name, so Python reads the hyphen as a minus sign:

In [0]:
# SUPPOSED TO FAIL -- AttributeError: 'DataFrame' object has no attribute 'Sub'
orders.Sub-Category.value_counts()

Python read that as `orders.Sub` minus `Category.value_counts()`, and failed on the first half. The error mentions a column called `Sub` that nobody ever created, which is why it is confusing the first time.

### 3.2 Two traps even with good names

Renaming fixes the hyphen problem, but the dot has two more traps that renaming does not fix.

**Trap 1: the name collides with a method.** A column called `count`, `size`, `index`, `values`, `mean`, or `name` loses to the DataFrame method with the same name.

In [0]:
stock = pd.DataFrame({"item": ["pens", "paper"], "count": [120, 40]})

print("stock['count']  ->", stock["count"].tolist())
print("stock.count     ->", type(stock.count))

The bracket gives you the column. The dot gives you a *method*, and `stock.count.sum()` would fail with a confusing message.

**Trap 2: you cannot create a column with a dot.** It looks like it works, then the column is not there:

In [0]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    stock.reorder = [False, True]          # looks like it creates a column

print("warning:", caught[0].message if caught else None)
print("is 'reorder' a column?", "reorder" in stock.columns)

So my rule: **the dot is fine for a quick look while exploring. In any code you keep, use brackets.** Brackets always mean "the column," for every name, and they work the same way for reading and for creating.

### 3.3 Rename everything once, right after loading

In [0]:
def to_snake(columns):
    'Lowercase, turn anything that is not a letter or digit into _, trim stray underscores.'
    return (pd.Index(columns)
            .str.strip()
            .str.lower()
            .str.replace(r"[^0-9a-z]+", "_", regex=True)
            .str.strip("_"))

original_names = orders.columns
orders.columns = to_snake(orders.columns)

pd.DataFrame({"before": original_names, "after": orders.columns})

Read `[^0-9a-z]+` as "one or more characters that are not a digit or a lowercase letter." It catches spaces, hyphens, slashes, parentheses, and `%` in one pattern, so the same function works on any dataset. That matters for your project: run `to_snake` on your file on day one and you never think about it again.

Two practical notes:

1. **Rename both tables before you merge.** If the orders say `region` and the People sheet still says `Region`, the merge key does not match and you get a `KeyError`. Section 4 renames `people` for exactly this reason.
2. **Keep the original names for your charts.** A chart axis labelled `sub_category` looks like code. Save the mapping and use it when you label:

In [0]:
pretty = dict(zip(orders.columns, original_names))
print(pretty["sub_category"], "|", pretty["order_priority"])

---
# 4. It ran. Did it work? Checking a merge

**From Scott's post:** *"The merge/join syntax. Didn't the guy who wrote this know SQL and why couldn't he use similar syntax."*

**From Matei's reply:** *"I agree that the merge syntax can be confusing, especially when first learning the difference between left and inner joins."*

The syntax is annoying, and Section 5 explains where it came from. But the syntax is not what makes merges dangerous. What makes them dangerous is that **a wrong merge almost never errors.** It runs, it returns a DataFrame, and the DataFrame has the wrong number of rows. You find out three weeks later when a total does not match.

So this section is about four checks. Run them every time you merge. They take one line each.

### 4.1 Left and inner, on data where it matters

Matei's question first. Join the athletes to the country lookup both ways and count:

In [0]:
athletes

In [0]:
noc_regions

In [0]:
athletes.merge(noc_regions, on="NOC", how="inner")

In [0]:
 athletes.merge(noc_regions, on="NOC", how="left")

In [0]:
inner = athletes.merge(noc_regions, on="NOC", how="inner")
left  = athletes.merge(noc_regions, on="NOC", how="left")

print("athlete rows:", len(athletes))
print("inner join:  ", len(inner))
print("left join:   ", len(left))

- **Inner** keeps only rows whose key appears in **both** tables. Three athletes vanished, with no warning.
- **Left** keeps **every** row of the left table. Rows with no match get `NaN` in the new columns.

Neither is wrong. They answer different questions. The problem with inner is that it answers its question *silently*: you have to count to notice anything happened. So the working habit is: **merge with `how="left"` first, look at what did not match, and only then decide whether to drop it.**

### 4.2 Check 1 and 2: count the rows, and ask pandas which rows matched

`indicator=True` adds a column called `_merge` that says, for each row, where it was found.

In [0]:
checked = athletes.merge(noc_regions, on="NOC", how="left", indicator=True)

print(checked["_merge"].value_counts())
print()
print("the keys that did not match:")
print(checked.loc[checked["_merge"] == "left_only", "NOC"].value_counts())

`left_only` means "in the athletes table, not in the lookup." The unmatched key is `SGP`.

That is exactly what happens in the real Olympics data. The athlete file codes Singapore as `SGP`, and `noc_regions.csv` lists it as `SIN`. On the full file an inner join drops 349 rows, all Singapore. If you did 2.3 with an inner join, Singapore is not in your results, and nothing told you.

Now the fix is a decision, and a small one: the two codes mean the same country, so correct the lookup and say so in a markdown cell.

In [0]:
noc_fixed = noc_regions.replace({"NOC": {"SIN": "SGP"}})

fixed = athletes.merge(noc_fixed, on="NOC", how="left", indicator=True)
print(fixed["_merge"].value_counts())

### 4.3 When the fix is not yours to make

The Superstore People sheet has the same kind of mismatch, and it is harder. First, give `people` the same snake_case names as `orders` (Section 3, note 1).

In [0]:
people.columns = to_snake(people.columns)

staffed = orders.merge(people, on="region", how="left", indicator=True)

print("rows before:", len(orders), " rows after:", len(staffed))
print(staffed["_merge"].value_counts())
print("\nunmatched regions in orders:  ", staffed.loc[staffed["_merge"] == "left_only", "region"].unique().tolist())
print("regions in people, not orders:", sorted(set(people["region"]) - set(orders["region"])))

The orders say `Canada`. The People sheet has `Eastern Canada` and `Western Canada`. In the real file that is 384 orders with no manager.

This time there is no correct one-line fix. Which of the two managers owns a `Canada` order? The orders table does not say. The options:

| Option | What it costs |
|---|---|
| Leave the manager blank for Canada, and say so | Honest. Canada drops out of any per-manager analysis. |
| Assign all of Canada to one manager | Invents a fact. One manager's numbers are inflated. |
| Split by province, if the orders have a state column | Possibly right, but only if you know which provinces each manager covers. That is a question for the data owner. |
| Inner join | The Canada orders disappear from *everything*, not only the per-manager view. The worst option, because it is silent. |

The first option is the right one for coursework. The general point: **a merge check tells you what did not match. It cannot tell you what to do about it.** Sometimes that is a code fix, and sometimes it is an email.

### 4.4 Check 3: say what the relationship should be, and let pandas enforce it

Every orders row should get **at most one** manager. That is a *many-to-one* merge: many orders, one manager per region. You can tell pandas that, and it will refuse to merge if the data disagrees.

Suppose the People sheet had been updated when Western Europe changed managers, and someone appended the new manager instead of replacing the old one:

In [0]:
people_dup = pd.concat(
    [people, pd.DataFrame({"person": ["Manager 99"], "region": ["Western Europe"]})],
    ignore_index=True,
)

exploded = orders.merge(people_dup, on="region", how="left")
print("rows before:", len(orders))
print("rows after: ", len(exploded), " <- more rows than we started with")

A left join gave us **more** rows than the left table had. Every Western Europe order matched two managers, so it now appears twice. Any sales total computed from `exploded` double-counts Western Europe, and the merge ran without a word.

`validate=` catches it:

In [0]:
# SUPPOSED TO FAIL -- MergeError: Merge keys are not unique in right dataset; not a many-to-one merge
orders.merge(people_dup, on="region", how="left", validate="many_to_one")

That is an error you *want*. It fails at the merge, where the cause is obvious, instead of three weeks later in a total.

The options are `"one_to_one"`, `"one_to_many"`, `"many_to_one"`, and `"many_to_many"`. For "add attributes from a lookup table," which is most merges you will write, the answer is `"many_to_one"`.

### 4.5 The merge checklist

| # | Check | Code |
|---|---|---|
| 1 | Row count before and after. For a left join onto a lookup, they must be **equal**. | `len(left)`, `len(result)` |
| 2 | Which rows matched? | `indicator=True`, then `result["_merge"].value_counts()` |
| 3 | Is the relationship what I think it is? | `validate="many_to_one"` |
| 4 | What exactly did not match? | `result.loc[result["_merge"] == "left_only", key].value_counts()` |

And one line to finish, once you have looked: `result = result.drop(columns="_merge")`.

For Scott and anyone else who thinks in SQL, the translation:

| SQL | pandas |
|---|---|
| `INNER JOIN` | `how="inner"` |
| `LEFT JOIN` | `how="left"` |
| `RIGHT JOIN` | `how="right"` |
| `FULL OUTER JOIN` | `how="outer"` |
| `ON a.k = b.k` | `on="k"` |
| `ON a.x = b.y` | `left_on="x", right_on="y"` |

#### Your Turn 4.1 - What would inner have cost?

Using `staffed` from 4.3, answer in numbers: if you had used `how="inner"`, how many orders and how much in `sales` would have disappeared from the result? Then write one sentence saying whether that is big enough to matter.

In [0]:
lost = staffed[staffed["_merge"] == "left_only"]

print("orders lost:", len(lost), "of", len(staffed), f"({len(lost) / len(staffed):.1%})")
print(f"sales lost:  {lost['sales'].sum():,.2f} of {staffed['sales'].sum():,.2f} "
      f"({lost['sales'].sum() / staffed['sales'].sum():.1%})")

<details>
<summary><b>Show answer</b></summary>

```python
lost = staffed[staffed["_merge"] == "left_only"]

print("orders lost:", len(lost), "of", len(staffed), f"({len(lost) / len(staffed):.1%})")
print(f"sales lost:  {lost['sales'].sum():,.2f} of {staffed['sales'].sum():,.2f} "
      f"({lost['sales'].sum() / staffed['sales'].sum():.1%})")
```

In this practice data Canada is one of ten regions, so about a tenth of everything. In the real Superstore file it is 384 of 51,290 orders, under 1%.

Whether that matters depends on the question, and that is the sentence I want you to write. For total company sales, under 1% is small. For a question about **Canada**, it is 100% of your data. The same missing rows can be trivial or fatal, and the number of rows alone does not tell you which.

</details>

---
# 5. `axis=0`, `axis=1`, and why `merge` feels bolted on

**From Scott's post:** *"I used this to do a bit of research on the two things that I dislike the most about pandas. 1. The merge/join syntax... 2. The axis=0 and axis=1 stuff... Both of these have apparently been fixed in a package called polars."*

Scott did the research, so this section is short. Both complaints are fair, both come from history, and both have a workaround you can use today.

### 5.1 Axis: one rule

**`axis` names the direction the operation travels.** `axis=0` travels down the rows, so you get one answer per column. `axis=1` travels across the columns, so you get one answer per row.

In [0]:
scores = pd.DataFrame({"q1": [8, 6, 9], "q2": [7, 5, 10]}, index=["Ana", "Ben", "Cy"])
scores

In [0]:
print(scores, "\n")
print("sum(axis=0), down the rows, one per column:")
print(scores.sum(axis=0), "\n")
print("sum(axis=1), across the columns, one per row:")
print(scores.sum(axis=1))

The confusing case is `drop`, where `axis=1` means "drop columns." It is the same rule (you are removing things that sit along the column direction), but nobody thinks of it that way. **The workaround is to not use `axis` there at all.** pandas accepts named arguments that say what they mean:

| Instead of | Write |
|---|---|
| `df.drop("col", axis=1)` | `df.drop(columns="col")` |
| `df.drop(3, axis=0)` | `df.drop(index=3)` |
| `df.rename(str.lower, axis=1)` | `df.rename(columns=str.lower)` |

You still need `axis` for `sum`, `mean`, `apply`, and friends. For those, remember "one answer per column" is the default.

### 5.2 Why merging feels like two different designs

Scott's history is right. pandas started in finance, where tables are time series lined up by their **index** (the row labels). The original way to combine tables was `df.join`, which lines up on the index. `merge`, which works on **columns** like SQL, came later. Both still exist, and they have different defaults:

| | `df.merge(other, ...)` | `df.join(other, ...)` |
|---|---|---|
| Matches on | columns you name | the **index** of `other` |
| Default `how` | `inner` | `left` |
| SQL-like? | yes | not really |

The default `how` is the dangerous difference: the same-sounding operation silently drops rows with one function and keeps them with the other.

**The workaround: always use `merge`, and always write `how=` out**, even when you want the default. In this course, write `on=`, `how=`, `validate=`, and `indicator=` on every merge while you are developing.

### 5.3 Polars

Scott mentioned polars, a newer DataFrame library that is much faster than pandas on big data and was designed without either quirk. It has no index, so there is only one kind of join, and it has no `axis`. For comparison:

```python
# pandas
orders.merge(people, on="region", how="left", validate="many_to_one")
orders.drop(columns="row_id")

# polars
orders.join(people, on="region", how="left", validate="m:1")
orders.drop("row_id")
```

You do not need it for this course. pandas is still what most employers, courses, and Stack Overflow answers use, and it is what you will be graded in. But it is worth knowing it exists, and worth trying if you ever hit a file pandas struggles with. Scott, if you try it on the HMDA data, I would like to hear how it goes.

---
# 6. Deciding what to change: the decision log

**From Jacob's post:** *"While challenging at some points, mainly the data transformation and deciding what should be changed or modified to be cleaner, it was great once I got the hang of it!"*

Jacob named the actual hard part. The code for a type conversion or a rename is one line. The hard part is deciding which one-liners to run, and it gets harder on your own project, where I have not planted the problems.

Three questions to ask **before** you change a column:

1. **Is it wrong, or just ugly?** `Sub-Category` is ugly; renaming it is housekeeping. `Postal Code` as a float is *wrong*; it has already destroyed data. Fix wrong things first, and spend your write-up on them.
2. **What am I going to do with this column?** A column you will never use does not need cleaning. It needs dropping, or ignoring. You do not have to clean all 24 Superstore columns. You have to clean the ones your question touches.
3. **What does the change destroy?** Every change costs something. Converting to a category makes new values an error. Dropping rows loses cases. Filling blanks invents values. If you cannot name the cost, you have not finished thinking about the change.

The habit that makes this manageable is a **decision log**: one row per decision, written as you go.

| Column | What I saw | What I did | What it cost |
|---|---|---|---|
| `postal_code` | Float; leading zeros lost; 80% blank | Converted to string, padded US codes to 5 digits | Nothing lost. Blanks stay blank: they mean non-US, not missing. |
| `order_priority` | Text, sorting alphabetically | Ordered category, Low < Medium < High < Critical | A new priority label would now raise an error until added. |
| `region` / People | `Canada` matches no manager | Left join, manager left blank for Canada | 384 orders excluded from per-manager results only. |
| All columns | Spaces and hyphens in names | `to_snake` | None. Original names kept for chart labels. |

That table is Section 4 of your final project notebook, and the rubric calls the last column the difference between bookkeeping and analysis. Start keeping one for your own dataset now. The assignment below asks for one.

---
# Where this leaves us

1. **`object` and `str` both mean text, depending on your pandas version.** On pandas 3, `object` means something is mixed, so check it with `.map(type).value_counts()`. On older pandas, run that check on every `object` column.
2. **"It has repeated values" is not a reason to use a category.** A small, *fixed* set of labels you group by is. Identifiers and free text are strings. When unsure, use string.
3. **Ordered categories are the best reason to use a category at all.** Tables and charts come out in the right order for free.
4. **Rename to snake_case once, right after loading, and use brackets in code you keep.** Rename both tables before merging.
5. **A bad merge does not error, so check every one.** Rows before and after, `indicator=True`, `validate=`, and look at what did not match.
6. **Use named arguments instead of `axis` where you can, and always write `how=` on a merge.**
7. **Name the cost of every change.** Jacob's hard part is the actual job.

---
# Assignment: Types, Names, and Joins on the Real Superstore File

Use `Global_Superstore_Orders_2012_2015.xlsx` from 2.3 and 2.4. Start a **new notebook**. Every step below needs code **and** a markdown cell with your answer in words.

**1. Load and measure.** Load both sheets (`Orders` and `People`). Print each sheet's shape, and the total memory of `Orders` with `.memory_usage(deep=True).sum()`.

**2. Rename.** Apply `to_snake` (copy it from Section 3.3) to both sheets. Print the before-and-after names for `Orders`.

**3. Type every column.** Make a markdown table with one row per `Orders` column: *column, dtype now, dtype you chose, reason.* Use the four questions from Section 2.6, and do not use "it has repeated values" as a reason. Then apply your choices in code. Requirements:
- `order_priority` must be an **ordered** category.
- `postal_code` must end up as text, and US codes must have their leading zeros back. (Hint: `.str.zfill(5)`, and look at what a float turns into when you convert it to a string first.)
- Print the memory again and report the change.

**4. Merge and check.** Merge `People` onto `Orders` with `how="left"`, `indicator=True`, and `validate="many_to_one"`. Report:
- row counts before and after,
- the `_merge` counts, and which region values did not match,
- how many orders, and how much `sales`, an **inner** join would have dropped.

Then, in markdown: what would you do about the unmatched rows, and what does that choice cost?

**5. Decision log.** A table like the one in Section 6, with at least four rows from steps 2 to 4.

**Before you submit:** *Kernel > Restart Kernel and Run All Cells*, and check that it runs top to bottom. Submit the `.ipynb`.